# Create a ***"dat_mnl_dct"***  that has ***"keys"*** = "data_col_lbls" and "***"rows"*** = "dtv"
1. ***"cstm_dat"*** is User defined list of ***"keys"*** from the ***"dat_mnl"***
2. Designed to be used in conjunction with ***"ib_dat_dct"*** to create diverse plotting dictionaries

## Daily Import of an XL workbook named ***dat_mnl***
1. manually copied from ""https://d.docs.live.net/AED59B3718F319AA/JL_2/dat_mnl/dat_mnl.xlsm"
2. to "\wsl.localhost\Ubuntu-20.04\home\ratlabs\JL_2\data\dat_mnl" 

## Def functions

In [182]:
# def import_dat_mnl():                                  #Fixed Path  Called Below Returns df_raw, df
import pandas as pd
from pathlib import Path

def import_dat_mnl():
    # Raw import (no header)
    xl_path = Path("/home/ratlabs/JL_2/data/dat_mnl/dat_mnl.xlsm")
    df_raw = pd.read_excel(
        xl_path,
        sheet_name="dat_mnl_main",
        header=None,
        engine="openpyxl"
    )

    # Detect the real header row by searching for "col_nms"
    header_row = df_raw.index[df_raw.eq("col_nms").any(axis=1)][0]

    # Promote that row to header
    df = df_raw.copy()
    df.columns = df.iloc[header_row].astype(str)

    # Drop all rows up to and including the header row
    df = df.drop(index=range(header_row + 1)).reset_index(drop=True)

    return df_raw, df



In [148]:
def fix_manual_df(df):
    """
    Converts a raw df where:
      - row 0 = column numbers
      - row 1 = actual column names
      - row 2+ = data
    into a clean dataframe with correct headers.
    """

    # Extract row 1 as header
    new_cols = df.iloc[1].tolist()

    # Apply new header
    df_fixed = df.copy()
    df_fixed.columns = new_cols

    # Drop row 0 and row 1
    df_fixed = df_fixed.drop(index=[0, 1]).reset_index(drop=True)

    return df_fixed


In [121]:
def create_plt_lst(dat_col_dct, ib_dct, plt_active_lst):
    """
    Filters dat_col_dict so that:
      - Only columns listed in plt_active_lst are included
      - Each column is filtered to rows whose dtv matches df_77_97_mrn['dtv']

    Returns:
        plt_active_dict : dict
            Keys = column names in plt_active_lst
            Values = filtered pandas Series aligned by dtv
    """

    # Extract the dtv values we want to keep
    dtv_filter_values = set(bi_dct["dtv"].unique())

    plt_active_dict = {}

    for col in plt_active_lst:
        if col not in dat_col_dict:
            continue  # skip missing columns safely

        series = dat_col_dict[col]

        # Filter the series by dtv alignment
        # Assumes dat_col_dict["dtv"] exists and is aligned row‑wise
        dtv_series = dat_col_dict["dtv"]

        filtered_series = series[dtv_series.isin(dtv_filter_values)]

        plt_active_dict[col] = filtered_series.reset_index(drop=True)

    return plt_active_dict


## importing the dat_mnl

## Building dat_mnl_dct

### Update ***"df_dat_mnl"*** from ***"XL"***

In [183]:
df_dat_mnlx,df_dat_mnl= import_dat_mnl()               # Load from XL 

In [185]:
# verify df_dat_mnl   #works

In [187]:
# verify type(df_dat_mnl)

In [190]:
# verify df_dat_mnl.columns.tolist     # Works

### Edit the ***"cstm_dat_mnl"*** list

In [192]:
df_dat_mnlx = fix_manual_df(df_dat_mnlx)        # Look to 2nd row for column names the numbers are in the first row



# Creating a plt_lst of dictionary of dat_mnl cols as the keys that that have data in rows "dtv"
1. matches "df_77_97_mrn" "dtv rows"
2. and contains desired "dat_mnl dat_cols"
3. It will be stored in "dat_mnl dict" pkl and used in plot def functions along with  "ib_dct" "dtv rows"

In [199]:
dat_col_dict = {col: df_dat_mnl[col] for col in df_dat_mnl.columns} # Calc the keys to the use for 


In [201]:
# verify list(dat_col_dict.keys())             # worked


In [202]:
# verify dat_col_dict    #worked

In [178]:
dtv = dat_col_dict["dtv"]
# verify   dtv  # woeked

In [203]:
notes = dat_col_dict["Notes"]

In [204]:
# verify 
notes           #works

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
788      _
789      _
790      _
791      _
792      _
Name: Notes, Length: 793, dtype: object

## Write the ***"dat_col_dict"*** to Pickle so it can be used to create plot by going down dictionaries

In [181]:
import pickle
with open("df_77_97_mrn.pkl", "rb") as f:  
    df_77_97_mrn = pickle.load(f)

In [96]:
# verify df_77_97_mrn #Works

In [97]:
plt_active_lst = ["dtv", "timestamp", "Notes"]  # 


In [98]:
plt_active_lst


['dtv', 'timestamp', 'Notes']

In [99]:
print(df_77_97_mrn.columns.tolist())


['timestamp', 'dtv', 'weight', 'vfa_(visceral_fat_area)', 'ecw/tbw', 'ecw/tbw_of_left_leg', 'bmr_(basal_metabolic_rate)', 'smm_(skeletal_muscle_mass)', 'khz-whole_body_phase_angle', 'whole_body_ecw/tbw_t_score', 'icw_(intracellular_water)']


In [100]:
plt_active_dict = create_plt_lst(dat_col_dict, df_77_97_mrn, plt_active_lst)


In [101]:
# verify plt_active_dict["dtv"]                    # worked
# verify  plt_active_dict["timestamp"]             # worked


In [102]:
for i, col in enumerate(df_77_97_mrn.columns):
    print(i, repr(col))


0 'timestamp'
1 'dtv'
2 'weight'
3 'vfa_(visceral_fat_area)'
4 'ecw/tbw'
5 'ecw/tbw_of_left_leg'
6 'bmr_(basal_metabolic_rate)'
7 'smm_(skeletal_muscle_mass)'
8 'khz-whole_body_phase_angle'
9 'whole_body_ecw/tbw_t_score'
10 'icw_(intracellular_water)'


In [ ]:
# This is the ready to plot list of combined 97 77 data of most interest
write_df_to_pickle(df_77_97_mrn, "df_77_97_mrn.pkl")
print("df_77_97_mrn written to pickle")
# verify df_77_97_mrn